In [ ]:
import logging
import pickle
import random
import warnings
from collections import defaultdict
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoConfig,
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    get_cosine_schedule_with_warmup,
)
from tqdm import tqdm

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)

In [ ]:
GENRES_DIR   = Path('/kaggle/input/jan-2026-dlgenai-project-messy-mashu/messy_mashup/genres_stems')
NOISE_DIR    = Path('/kaggle/input/jan-2026-dlgenai-project-messy-mashu/messy_mashup/ESC-50-master/audio')
MASHUP_DIR   = Path('/kaggle/input/jan-2026-dlgenai-project-messy-mashu/messy_mashup/mashups')
TEST_CSV     = Path('/kaggle/input/jan-2026-dlgenai-project-messy-mashu/messy_mashup/test.csv')
CKPT_DIR     = Path('/kaggle/working/best_model')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME   = 'MIT/ast-finetuned-audioset-10-10-0.4593'
SAMPLE_RATE  = 16000
DURATION_SEC = 10
TARGET_LEN   = SAMPLE_RATE * DURATION_SEC
STEMS        = ['bass', 'drums', 'other', 'vocals']

BATCH_SIZE         = 8
GRAD_ACCUM         = 2
NUM_EPOCHS         = 60
LR                 = 3e-5
WEIGHT_DECAY       = 0.01
PATIENCE           = 10
SEED               = 42
LABEL_SMOOTHING    = 0.1
WARMUP_RATIO       = 0.1
AUGMENT_MULTIPLIER = 8
UNFREEZE_EPOCH     = 5
CROSS_MIX_PROB     = 0.9
TEMPO_PERTURB      = True
TEMPO_RANGE        = (0.85, 1.15)
NOISE_PROB         = 0.8
NOISE_SNR_RANGE    = (5, 25)
PITCH_SHIFT        = True
PITCH_SEMITONES    = (-2, 2)
TTA_RUNS           = 15

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
def fix_len(y, target=TARGET_LEN):
    if len(y) < target:
        return np.pad(y, (0, target - len(y)))
    return y[:target]

def load_stem(path):
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True, duration=DURATION_SEC)
    return fix_len(y).astype(np.float32)

def time_stretch(y, rate):
    try:
        y = librosa.effects.time_stretch(y, rate=rate)
    except Exception:
        pass
    return fix_len(y)

def pitch_shift(y, n_steps):
    try:
        y = librosa.effects.pitch_shift(y, sr=SAMPLE_RATE, n_steps=n_steps)
    except Exception:
        pass
    return fix_len(y)

def add_noise(y, noise_files, snr_db):
    if not noise_files:
        return y
    try:
        noise, _ = librosa.load(random.choice(noise_files), sr=SAMPLE_RATE, mono=True)
    except Exception:
        return y
    noise = fix_len(noise)
    sig_pow   = np.mean(y ** 2) + 1e-10
    noise_pow = np.mean(noise ** 2) + 1e-10
    noise    *= np.sqrt(sig_pow / (noise_pow * 10 ** (snr_db / 10)))
    mixed = y + noise
    return (mixed / (np.max(np.abs(mixed)) + 1e-8)).astype(np.float32)

def random_gain(y, low=0.7, high=1.0):
    return y * random.uniform(low, high)

def random_offset(y):
    offset = random.randint(0, TARGET_LEN // 4)
    return fix_len(y[offset:])

In [ ]:
def build_stem_index(genres_dir):
    index = defaultdict(lambda: defaultdict(list))
    for genre_dir in sorted(genres_dir.iterdir()):
        if not genre_dir.is_dir():
            continue
        for track_dir in sorted(genre_dir.iterdir()):
            if not track_dir.is_dir():
                continue
            for stem in STEMS:
                p = track_dir / f'{stem}.wav'
                if p.exists():
                    index[genre_dir.name][stem].append(p)
    return index

def build_records(genres_dir):
    records = []
    for genre_dir in sorted(genres_dir.iterdir()):
        if not genre_dir.is_dir():
            continue
        for track_dir in sorted(genre_dir.iterdir()):
            if not track_dir.is_dir():
                continue
            if all((track_dir / f'{s}.wav').exists() for s in STEMS):
                records.append({'genre': genre_dir.name, 'track_dir': str(track_dir)})
    return records

stem_index  = build_stem_index(GENRES_DIR)
noise_files = sorted(NOISE_DIR.glob('*.wav')) if NOISE_DIR.exists() else []
records     = build_records(GENRES_DIR)
print(f'Tracks: {len(records)}  Noise files: {len(noise_files)}')

In [ ]:
df = pd.DataFrame(records)
le = LabelEncoder()
df['label'] = le.fit_transform(df['genre'])
label2idx = {g: i for i, g in enumerate(le.classes_)}
idx2label = {i: g for g, i in label2idx.items()}
num_classes = len(label2idx)
print(f'Classes ({num_classes}): {list(le.classes_)}')

train_df, val_df = train_test_split(df, test_size=0.10, stratify=df['label'], random_state=SEED)
train_records = train_df[['genre', 'label', 'track_dir']].to_dict('records')
val_records   = val_df[['genre', 'label', 'track_dir']].to_dict('records')
print(f'Train: {len(train_records)}  Val: {len(val_records)}')

In [ ]:
def make_augmented_mix(genre, stem_index, noise_files):
    stems_audio = []
    for stem in STEMS:
        candidates = stem_index[genre][stem]
        if not candidates:
            stems_audio.append(np.zeros(TARGET_LEN, dtype=np.float32))
            continue
        y = load_stem(random.choice(candidates))
        y = random_offset(y)
        if TEMPO_PERTURB and random.random() < 0.7:
            y = time_stretch(y, random.uniform(*TEMPO_RANGE))
        if PITCH_SHIFT and random.random() < 0.4:
            y = pitch_shift(y, random.uniform(*PITCH_SEMITONES))
        y = random_gain(y, 0.6, 1.0)
        stems_audio.append(y)
    weights = np.array([random.uniform(0.5, 1.0) for _ in STEMS])
    weights /= weights.sum()
    mix = sum(w * s for w, s in zip(weights, stems_audio))
    mix = mix / (np.max(np.abs(mix)) + 1e-8)
    if noise_files and random.random() < NOISE_PROB:
        mix = add_noise(mix, noise_files, random.uniform(*NOISE_SNR_RANGE))
    return mix.astype(np.float32)

def make_clean_mix(track_dir):
    track_path = Path(track_dir)
    stems_audio = []
    for stem in STEMS:
        p = track_path / f'{stem}.wav'
        y = load_stem(p) if p.exists() else np.zeros(TARGET_LEN, dtype=np.float32)
        stems_audio.append(y)
    mix = np.mean(stems_audio, axis=0)
    return (mix / (np.max(np.abs(mix)) + 1e-8)).astype(np.float32)

In [ ]:
class TrainAudioDataset(Dataset):
    def __init__(self, records, stem_index, noise_files, feature_extractor, multiplier=AUGMENT_MULTIPLIER):
        self.records = records
        self.stem_index = stem_index
        self.noise_files = noise_files
        self.feature_extractor = feature_extractor
        self._index = list(range(len(records))) * multiplier

    def __len__(self):
        return len(self._index)

    def __getitem__(self, idx):
        rec   = self.records[self._index[idx]]
        audio = make_augmented_mix(rec['genre'], self.stem_index, self.noise_files)
        inputs = self.feature_extractor(audio, sampling_rate=SAMPLE_RATE, return_tensors='pt')
        return {k: v.squeeze(0) for k, v in inputs.items()}, rec['label']


class ValAudioDataset(Dataset):
    def __init__(self, records, feature_extractor):
        self.records = records
        self.feature_extractor = feature_extractor

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec   = self.records[idx]
        audio = make_clean_mix(rec['track_dir'])
        inputs = self.feature_extractor(audio, sampling_rate=SAMPLE_RATE, return_tensors='pt')
        return {k: v.squeeze(0) for k, v in inputs.items()}, rec['label']


def collate_fn(batch):
    inputs_list, labels = zip(*batch)
    collated = {key: torch.stack([x[key] for x in inputs_list]) for key in inputs_list[0]}
    return collated, torch.tensor(labels, dtype=torch.long)

In [ ]:
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(MODEL_NAME)
config.num_labels = num_classes
config.label2id   = label2idx
config.id2label   = idx2label

model = AutoModelForAudioClassification.from_pretrained(
    MODEL_NAME, config=config, ignore_mismatched_sizes=True
).to(device)

for name, param in model.named_parameters():
    if 'classifier' not in name and 'layernorm' not in name:
        param.requires_grad = False

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Backbone frozen. Trainable params: {n_trainable:,}')

In [ ]:
train_ds = TrainAudioDataset(train_records, stem_index, noise_files, feature_extractor)
val_ds   = ValAudioDataset(val_records, feature_extractor)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR * 5, weight_decay=WEIGHT_DECAY
)
total_steps  = (len(train_loader) // GRAD_ACCUM) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
scaler = torch.cuda.amp.GradScaler()

In [ ]:
best_f1, patience_counter, best_epoch = 0.0, 0, 0

for epoch in range(1, NUM_EPOCHS + 1):

    if epoch == UNFREEZE_EPOCH:
        for param in model.parameters():
            param.requires_grad = True
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        remaining_steps = (len(train_loader) // GRAD_ACCUM) * (NUM_EPOCHS - epoch + 1)
        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(remaining_steps * 0.05),
            num_training_steps=remaining_steps,
        )
        n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'All layers unfrozen. Trainable params: {n_trainable:,}')

    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []
    optimizer.zero_grad()

    for i, (batch_inputs, labels) in enumerate(tqdm(train_loader, desc=f'Epoch {epoch} [train]', leave=False)):
        batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
        labels = labels.to(device)
        with torch.cuda.amp.autocast():
            outputs = model(**batch_inputs)
            loss = criterion(outputs.logits, labels) / GRAD_ACCUM
        scaler.scale(loss).backward()
        if (i + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
        total_loss += loss.item() * GRAD_ACCUM
        all_preds.extend(outputs.logits.argmax(1).detach().cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    train_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for batch_inputs, labels in tqdm(val_loader, desc='val', leave=False):
            batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
            with torch.cuda.amp.autocast():
                outputs = model(**batch_inputs)
            val_preds.extend(outputs.logits.argmax(1).cpu().tolist())
            val_labels.extend(labels.tolist())

    val_acc = accuracy_score(val_labels, val_preds)
    val_f1  = f1_score(val_labels, val_preds, average='macro', zero_division=0)

    print(f'Epoch {epoch:3d} | train_f1={train_f1:.4f} | val_acc={val_acc:.4f} val_f1={val_f1:.4f}')

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_epoch = epoch
        patience_counter = 0
        model.save_pretrained(CKPT_DIR)
        feature_extractor.save_pretrained(CKPT_DIR)
        with open(CKPT_DIR / 'label_encoder.pkl', 'wb') as f:
            pickle.dump(le, f)
        print(f'  -> New best F1={best_f1:.4f} saved.')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch}.')
            break

print(f'Training done. Best val F1={best_f1:.4f} at epoch {best_epoch}')

In [ ]:
print('Classification report on val set (clean mixes):')
best_model = AutoModelForAudioClassification.from_pretrained(str(CKPT_DIR)).to(device)
best_model.eval()

val_preds, val_labels = [], []
with torch.no_grad():
    for batch_inputs, labels in tqdm(val_loader, desc='Val eval'):
        batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
        with torch.cuda.amp.autocast():
            outputs = best_model(**batch_inputs)
        val_preds.extend(outputs.logits.argmax(1).cpu().tolist())
        val_labels.extend(labels.tolist())

pred_names  = [idx2label[p] for p in val_preds]
label_names = [idx2label[l] for l in val_labels]
print(classification_report(label_names, pred_names, target_names=sorted(label2idx)))

In [ ]:
def augment_audio(y, noise_files):
    offset = random.randint(0, TARGET_LEN // 4)
    y = fix_len(y[offset:])
    if random.random() < 0.7:
        try:
            y = librosa.effects.time_stretch(y, rate=random.uniform(*TEMPO_RANGE))
        except Exception:
            pass
        y = fix_len(y)
    if random.random() < 0.4:
        try:
            y = librosa.effects.pitch_shift(y, sr=SAMPLE_RATE, n_steps=random.uniform(*PITCH_SEMITONES))
        except Exception:
            pass
        y = fix_len(y)
    y = y * random.uniform(0.7, 1.0)
    if noise_files and random.random() < NOISE_PROB:
        snr_db = random.uniform(*NOISE_SNR_RANGE)
        try:
            noise, _ = librosa.load(random.choice(noise_files), sr=SAMPLE_RATE, mono=True)
            noise     = fix_len(noise)
            sig_pow   = np.mean(y ** 2) + 1e-10
            noise_pow = np.mean(noise ** 2) + 1e-10
            noise    *= np.sqrt(sig_pow / (noise_pow * 10 ** (snr_db / 10)))
            y         = y + noise
        except Exception:
            pass
    return (y / (np.max(np.abs(y)) + 1e-8)).astype(np.float32)


test_df  = pd.read_csv(TEST_CSV)
song_ids = test_df['filename'].apply(lambda x: Path(x).stem).tolist()
print(f'Test samples: {len(test_df)}')

all_audio = []
for sid in tqdm(song_ids, desc='Loading test audio'):
    p = MASHUP_DIR / f'{sid}.wav'
    if p.exists():
        y, _ = librosa.load(p, sr=SAMPLE_RATE, mono=True, duration=DURATION_SEC)
        all_audio.append(fix_len(y).astype(np.float32))
    else:
        all_audio.append(np.zeros(TARGET_LEN, dtype=np.float32))

print(f'Loaded {len(all_audio)} audio clips')

In [ ]:
best_model.eval()
grand_sum = None
TTA_BATCH = 8

for run_idx in range(TTA_RUNS):
    is_clean = (run_idx == 0)
    run_logits = []
    for start in tqdm(range(0, len(all_audio), TTA_BATCH),
                      desc=f'TTA {"clean" if is_clean else run_idx}/{TTA_RUNS-1}', leave=False):
        batch = all_audio[start: start + TTA_BATCH]
        processed = list(batch) if is_clean else [augment_audio(a.copy(), noise_files) for a in batch]
        inputs = feature_extractor(processed, sampling_rate=SAMPLE_RATE, return_tensors='pt', padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad(), torch.cuda.amp.autocast():
            outputs = best_model(**inputs)
        run_logits.append(outputs.logits.cpu().float())
    run_tensor = torch.cat(run_logits, dim=0)
    grand_sum  = run_tensor if grand_sum is None else grand_sum + run_tensor
    print(f'TTA run {run_idx+1}/{TTA_RUNS} done')

avg_logits  = grand_sum / TTA_RUNS
final_preds = avg_logits.argmax(dim=1).tolist()
predictions = [idx2label[p] for p in final_preds]
print('TTA inference complete')

In [ ]:
out_df = pd.DataFrame({
    'id':    test_df['id'].astype(str).str.zfill(4).tolist(),
    'genre': predictions,
})
out_df.to_csv('/kaggle/working/submission.csv', index=False)

print(f'submission.csv saved: {len(out_df)} rows')
print(out_df['genre'].value_counts().to_string())
out_df.head(10)